In [1]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

KeyboardInterrupt: 

In [ ]:
pip uninstall torch -y
pip install torch==2.1.0 --index-url https://download.pytorch.org/whl/cu118

SyntaxError: invalid syntax (112339395.py, line 1)

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.0 GB


In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# تحميل الداتا
df = pd.read_csv('../data/arabic/fake_news_data.csv')
df['label'] = df['Label'].map({'real': 0, 'fake': 1})
df = df[['Article_content', 'label']].dropna()

print(f"Total: {len(df)}")
print(df['label'].value_counts())

c:\Users\User\OneDrive\Desktop\fake-news-detector\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total: 46097
label
1    42243
0     3854
Name: count, dtype: int64


In [ ]:
from sklearn.utils import resample

# معالجة الـ imbalance
df_real = df[df['label'] == 0]
df_fake = df[df['label'] == 1].sample(n=len(df_real)*2, random_state=42)
df_balanced = pd.concat([df_real, df_fake]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Balanced: {len(df_balanced)}")
print(df_balanced['label'].value_counts())

# Tokenizer
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Done!")

Balanced: 11562
label
1    7708
0    3854
Name: count, dtype: int64

Loading tokenizer...
Done!


In [ ]:
class ArabicNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.labels = list(labels)
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_balanced['Article_content'],
    df_balanced['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_balanced['label']
)

print("Tokenizing...")
train_dataset = ArabicNewsDataset(train_texts, train_labels, tokenizer)
val_dataset   = ArabicNewsDataset(val_texts,   val_labels,   tokenizer)

# batch_size صغير عشان الـ 6GB
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Tokenizing...
Train: 9249 | Val: 2313


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# تحميل الموديل
print("Loading AraBERT...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)
print("Model loaded!")

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for i, batch in enumerate(loader):
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if i % 100 == 0:
            print(f"  Step {i}/{len(loader)} - Loss: {loss.item():.4f}")
    return total_loss / len(loader)

def eval_epoch(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    return acc, f1

# Train
EPOCHS = 3
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss = train_epoch(model, train_loader, optimizer)
    val_acc, val_f1 = eval_epoch(model, val_loader)
    print(f"Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

print("\nTraining Done! 🎉")

Training on: cuda
Loading AraBERT...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded!

Epoch 1/3
  Step 0/1157 - Loss: 0.5942
  Step 100/1157 - Loss: 0.1061
  Step 200/1157 - Loss: 0.0173
  Step 300/1157 - Loss: 0.0353
  Step 400/1157 - Loss: 0.0042
  Step 500/1157 - Loss: 0.0037
  Step 600/1157 - Loss: 0.0029
  Step 700/1157 - Loss: 0.0023
  Step 800/1157 - Loss: 0.0006
  Step 900/1157 - Loss: 0.0014
  Step 1000/1157 - Loss: 0.0035
  Step 1100/1157 - Loss: 0.0065
Loss: 0.0438 | Val Acc: 0.9952 | Val F1: 0.9952

Epoch 2/3
  Step 0/1157 - Loss: 0.0054
  Step 100/1157 - Loss: 0.0046
  Step 200/1157 - Loss: 0.0006
  Step 300/1157 - Loss: 0.0005
  Step 400/1157 - Loss: 0.0008
  Step 500/1157 - Loss: 0.0012
  Step 600/1157 - Loss: 0.0002
  Step 700/1157 - Loss: 0.0013
  Step 800/1157 - Loss: 0.0005
  Step 900/1157 - Loss: 0.0005
  Step 1000/1157 - Loss: 0.0087
  Step 1100/1157 - Loss: 0.0001
Loss: 0.0082 | Val Acc: 0.9974 | Val F1: 0.9974

Epoch 3/3
  Step 0/1157 - Loss: 0.0003
  Step 100/1157 - Loss: 0.0001
  Step 200/1157 - Loss: 0.0003
  Step 300/1157 - Loss

In [ ]:
import os

save_path = "../models/arabert_arabic"
os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Model saved to {save_path}")
print(f"Files: {os.listdir(save_path)}")

✅ Model saved to ../models/arabert_arabic
Files: ['config.json', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.txt']


In [ ]:
# تحميل LIAR dataset للإنجليزي
import pandas as pd

# LIAR dataset
train = pd.read_csv('https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/train.tsv', 
                    sep='\t', header=None)
print(train.shape)
print(train.head())

(10240, 14)
           0            1                                                  2   \
0   2635.json        false  Says the Annies List political group supports ...   
1  10540.json    half-true  When did the decline of coal start? It started...   
2    324.json  mostly-true  Hillary Clinton agrees with John McCain "by vo...   
3   1123.json        false  Health care reform legislation is likely to ma...   
4   9028.json    half-true  The economic turnaround started at the end of ...   

                                   3               4                     5   \
0                            abortion    dwayne-bohac  State representative   
1  energy,history,job-accomplishments  scott-surovell        State delegate   
2                      foreign-policy    barack-obama             President   
3                         health-care    blog-posting                   NaN   
4                        economy,jobs   charlie-crist                   NaN   

         6           7    

In [ ]:
# تجهيز الداتا
df_en = train[[1, 2]].copy()
df_en.columns = ['label', 'text']

# تحويل الـ labels لـ binary (fake/real)
fake_labels = ['false', 'barely-true', 'pants-fire']
real_labels = ['true', 'mostly-true', 'half-true']

df_en = df_en[df_en['label'].isin(fake_labels + real_labels)]
df_en['label'] = df_en['label'].apply(lambda x: 1 if x in fake_labels else 0)
df_en = df_en.dropna()

print(f"Total: {len(df_en)}")
print(df_en['label'].value_counts())

# Balance
df_fake_en = df_en[df_en['label'] == 1]
df_real_en = df_en[df_en['label'] == 0]
min_size = min(len(df_fake_en), len(df_real_en))
df_en_balanced = pd.concat([
    df_fake_en.sample(n=min_size, random_state=42),
    df_real_en.sample(n=min_size, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nBalanced: {len(df_en_balanced)}")
print(df_en_balanced['label'].value_counts())

# Tokenizer
EN_MODEL = "roberta-base"
print("\nLoading RoBERTa tokenizer...")
tokenizer_en = AutoTokenizer.from_pretrained(EN_MODEL)
print("Done!")

Total: 10240
label
0    5752
1    4488
Name: count, dtype: int64

Balanced: 8976
label
1    4488
0    4488
Name: count, dtype: int64

Loading RoBERTa tokenizer...
Done!


In [ ]:
class EnglishNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.labels = list(labels)
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Split
train_texts_en, val_texts_en, train_labels_en, val_labels_en = train_test_split(
    df_en_balanced['text'],
    df_en_balanced['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_en_balanced['label']
)

print("Tokenizing...")
train_dataset_en = EnglishNewsDataset(train_texts_en, train_labels_en, tokenizer_en)
val_dataset_en   = EnglishNewsDataset(val_texts_en,   val_labels_en,   tokenizer_en)

train_loader_en = DataLoader(train_dataset_en, batch_size=8, shuffle=True)
val_loader_en   = DataLoader(val_dataset_en,   batch_size=16)

print(f"Train: {len(train_dataset_en)} | Val: {len(val_dataset_en)}")

Tokenizing...
Train: 7180 | Val: 1796


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

print("Loading RoBERTa model...")
model_en = AutoModelForSequenceClassification.from_pretrained(EN_MODEL, num_labels=2)
model_en.to(device)
print("Model loaded!")

optimizer_en = AdamW(model_en.parameters(), lr=2e-5, weight_decay=0.01)

EPOCHS = 3
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    # Train
    model_en.train()
    total_loss = 0
    for i, batch in enumerate(train_loader_en):
        optimizer_en.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model_en(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer_en.step()
        total_loss += outputs.loss.item()
        if i % 100 == 0:
            print(f"  Step {i}/{len(train_loader_en)} - Loss: {outputs.loss.item():.4f}")
    
    # Eval
    model_en.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader_en:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model_en(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    print(f"Loss: {total_loss/len(train_loader_en):.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}")

print("\nTraining Done! 🎉")

Training on: cuda
Loading RoBERTa model...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.out_proj.bias', 'classifier.out_proj.weight', 'classifier.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded!

Epoch 1/3
  Step 0/898 - Loss: 0.7084
  Step 100/898 - Loss: 0.6901
  Step 200/898 - Loss: 0.7169
  Step 300/898 - Loss: 0.6006
  Step 400/898 - Loss: 0.7360
  Step 500/898 - Loss: 0.6999
  Step 600/898 - Loss: 0.7148
  Step 700/898 - Loss: 0.6635
  Step 800/898 - Loss: 0.6458
Loss: 0.6853 | Val Acc: 0.5000 | Val F1: 0.3333

Epoch 2/3
  Step 0/898 - Loss: 0.6843
  Step 100/898 - Loss: 0.7060
  Step 200/898 - Loss: 0.7216
  Step 300/898 - Loss: 0.6893
  Step 400/898 - Loss: 0.7384
  Step 500/898 - Loss: 0.7724
  Step 600/898 - Loss: 0.8098
  Step 700/898 - Loss: 0.6375
  Step 800/898 - Loss: 0.7173
Loss: 0.6636 | Val Acc: 0.5818 | Val F1: 0.5565

Epoch 3/3
  Step 0/898 - Loss: 0.4305
  Step 100/898 - Loss: 0.5765
  Step 200/898 - Loss: 0.5021
  Step 300/898 - Loss: 0.7769
  Step 400/898 - Loss: 0.4402
  Step 500/898 - Loss: 0.4077
  Step 600/898 - Loss: 0.4371
  Step 700/898 - Loss: 0.5663
  Step 800/898 - Loss: 0.6772
Loss: 0.6250 | Val Acc: 0.6269 | Val F1: 0.6261

Trai

In [ ]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

EN_MODEL = "roberta-base"
print("Loading RoBERTa tokenizer...")
tokenizer_en = AutoTokenizer.from_pretrained(EN_MODEL)
print("Done!")

c:\Users\User\OneDrive\Desktop\fake-news-detector\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading RoBERTa tokenizer...
Done!


In [ ]:
train = pd.read_csv('https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/train.tsv', 
                    sep='\t', header=None)
df_en = train[[1, 2]].copy()
df_en.columns = ['label', 'text']
fake_labels = ['false', 'barely-true', 'pants-fire']
real_labels = ['true', 'mostly-true', 'half-true']
df_en = df_en[df_en['label'].isin(fake_labels + real_labels)]
df_en['label'] = df_en['label'].apply(lambda x: 1 if x in fake_labels else 0)
df_en = df_en.dropna()
df_fake_en = df_en[df_en['label'] == 1]
df_real_en = df_en[df_en['label'] == 0]
min_size = min(len(df_fake_en), len(df_real_en))
df_en_balanced = pd.concat([
    df_fake_en.sample(n=min_size, random_state=42),
    df_real_en.sample(n=min_size, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Balanced: {len(df_en_balanced)}")

Balanced: 8976


In [ ]:
EN_MODEL = "roberta-base"
tokenizer_en = AutoTokenizer.from_pretrained(EN_MODEL)

class EnglishNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.labels = list(labels)
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True,
            max_length=max_len, return_tensors="pt"
        )
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_texts_en, val_texts_en, train_labels_en, val_labels_en = train_test_split(
    df_en_balanced['text'], df_en_balanced['label'],
    test_size=0.2, random_state=42, stratify=df_en_balanced['label']
)
train_dataset_en = EnglishNewsDataset(train_texts_en, train_labels_en, tokenizer_en)
val_dataset_en   = EnglishNewsDataset(val_texts_en,   val_labels_en,   tokenizer_en)
train_loader_en = DataLoader(train_dataset_en, batch_size=8, shuffle=True)
val_loader_en   = DataLoader(val_dataset_en,   batch_size=16)
print(f"Train: {len(train_dataset_en)} | Val: {len(val_dataset_en)}")

Train: 7180 | Val: 1796


In [6]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
print("Imports OK!")

c:\Users\User\OneDrive\Desktop\fake-news-detector\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK!


In [7]:
train = pd.read_csv('https://raw.githubusercontent.com/thiagorainmaker77/liar_dataset/master/train.tsv', 
                    sep='\t', header=None)
df_en = train[[1, 2]].copy()
df_en.columns = ['label', 'text']
fake_labels = ['false', 'barely-true', 'pants-fire']
real_labels = ['true', 'mostly-true', 'half-true']
df_en = df_en[df_en['label'].isin(fake_labels + real_labels)]
df_en['label'] = df_en['label'].apply(lambda x: 1 if x in fake_labels else 0)
df_en = df_en.dropna()
df_fake_en = df_en[df_en['label'] == 1]
df_real_en = df_en[df_en['label'] == 0]
min_size = min(len(df_fake_en), len(df_real_en))
df_en_balanced = pd.concat([
    df_fake_en.sample(n=min_size, random_state=42),
    df_real_en.sample(n=min_size, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Balanced: {len(df_en_balanced)}")

Balanced: 8976


In [8]:
EN_MODEL = "roberta-base"
tokenizer_en = AutoTokenizer.from_pretrained(EN_MODEL)

class EnglishNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.labels = list(labels)
        self.encodings = tokenizer(
            list(texts), truncation=True, padding=True,
            max_length=max_len, return_tensors="pt"
        )
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_texts_en, val_texts_en, train_labels_en, val_labels_en = train_test_split(
    df_en_balanced['text'], df_en_balanced['label'],
    test_size=0.2, random_state=42, stratify=df_en_balanced['label']
)
train_dataset_en = EnglishNewsDataset(train_texts_en, train_labels_en, tokenizer_en)
val_dataset_en   = EnglishNewsDataset(val_texts_en,   val_labels_en,   tokenizer_en)
train_loader_en = DataLoader(train_dataset_en, batch_size=8, shuffle=True)
val_loader_en   = DataLoader(val_dataset_en,   batch_size=16)
print(f"Train: {len(train_dataset_en)} | Val: {len(val_dataset_en)}")

Train: 7180 | Val: 1796


In [9]:
class EnglishNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.labels = list(labels)
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt"
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Split
train_texts_en, val_texts_en, train_labels_en, val_labels_en = train_test_split(
    df_en_balanced['text'],
    df_en_balanced['label'],
    test_size=0.2,
    random_state=42,
    stratify=df_en_balanced['label']
)

print("Tokenizing...")
train_dataset_en = EnglishNewsDataset(train_texts_en, train_labels_en, tokenizer_en)
val_dataset_en   = EnglishNewsDataset(val_texts_en,   val_labels_en,   tokenizer_en)

train_loader_en = DataLoader(train_dataset_en, batch_size=8, shuffle=True)
val_loader_en   = DataLoader(val_dataset_en,   batch_size=16)

print(f"Train: {len(train_dataset_en)} | Val: {len(val_dataset_en)}")

Tokenizing...
Train: 7180 | Val: 1796


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

print("Loading RoBERTa model...")
model_en = AutoModelForSequenceClassification.from_pretrained(EN_MODEL, num_labels=2)
model_en.to(device)
print("Model loaded!")

optimizer_en = AdamW(model_en.parameters(), lr=2e-5, weight_decay=0.01)

EPOCHS = 3
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    
    # Train
    model_en.train()
    total_loss = 0
    for i, batch in enumerate(train_loader_en):
        optimizer_en.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model_en(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward()
        optimizer_en.step()
        total_loss += outputs.loss.item()
        if i % 100 == 0:
            print(f"  Step {i}/{len(train_loader_en)} - Loss: {outputs.loss.item():.4f}")
    
    # Eval
    model_en.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader_en:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs = model_en(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='weighted')
    print(f"Loss: {total_loss/len(train_loader_en):.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}")

print("\nTraining Done! 🎉")

Training on: cuda
Loading RoBERTa model...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.out_proj.weight', 'classifier.dense.bias', 'classifier.out_proj.bias', 'classifier.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded!

Epoch 1/3
  Step 0/898 - Loss: 0.7376
  Step 100/898 - Loss: 0.6923
  Step 200/898 - Loss: 0.6830
  Step 300/898 - Loss: 0.8339
  Step 400/898 - Loss: 0.7549
  Step 500/898 - Loss: 0.9544
  Step 600/898 - Loss: 0.6001
  Step 700/898 - Loss: 0.7234
  Step 800/898 - Loss: 0.6735
Loss: 0.6761 | Val Acc: 0.6102 | Val F1: 0.6032

Epoch 2/3
  Step 0/898 - Loss: 0.6069
  Step 100/898 - Loss: 0.6998
  Step 200/898 - Loss: 0.5814
  Step 300/898 - Loss: 0.8412
  Step 400/898 - Loss: 0.5388
  Step 500/898 - Loss: 0.6680
  Step 600/898 - Loss: 0.7187
  Step 700/898 - Loss: 0.7169
  Step 800/898 - Loss: 0.5778
Loss: 0.6459 | Val Acc: 0.6253 | Val F1: 0.6234

Epoch 3/3
  Step 0/898 - Loss: 0.5170
  Step 100/898 - Loss: 0.4954
  Step 200/898 - Loss: 0.9285
  Step 300/898 - Loss: 0.6151
  Step 400/898 - Loss: 0.8923
  Step 500/898 - Loss: 0.7602
  Step 600/898 - Loss: 0.6249
  Step 700/898 - Loss: 0.5945
  Step 800/898 - Loss: 0.5801
Loss: 0.5886 | Val Acc: 0.6208 | Val F1: 0.6147

Trai

In [12]:
import os

save_path_en = "../models/roberta_english"
os.makedirs(save_path_en, exist_ok=True)

model_en.save_pretrained(save_path_en)
tokenizer_en.save_pretrained(save_path_en)

print(f"✅ RoBERTa saved!")
print(f"Files: {os.listdir(save_path_en)}")

✅ RoBERTa saved!
Files: ['config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


In [2]:
import importlib
import torch
# تأكد إن الـ version صح
print(torch.__version__)
print(torch.cuda.is_available())

2.1.0+cu118
True


In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.1.0+cu118
True
